# 공연·외식업 분석 독립 검토 종합

2026년 10월 5일 검토. 13~15번의 독립 감사 결과를 읽어 종합 보고서를 생성한다. 원래 분석과 원자료는 변경하지 않는다. 확정된 변수 해석 문제, 자료로 확인한 설계 한계, 아직 검증하지 않은 가능성을 구분한다.

최종 전달 형식은 오프라인 HTML 기술 검토 보고서다. 정확한 비교와 감사 항목 확인이 목적이므로 표를 사용한다. 차트의 시각적 추세보다 개수·분모·확정 여부를 정확히 전달하기 위한 선택이다. 보고서 구조는 요약 → 범위·정의 → 항목별 근거 → 독립 재계산 → 수정 우선순위 → 남은 질문이다.

그림 계약: 점포 규모에 따라 매출 제공률이 얼마나 다른가를 2023~2025년 상권×업종×분기 7개 규모 구간으로 비교한다. 단일 계열 세로 막대, 공유 렌더러의 단일 청색 계열, y축 0~100%이며 단위는 퍼센트 숫자다. 점포0과 점포미제공은 그림에서 제외하고 상세 표에 남긴다. 전체 폭 HTML 그림과 표를 함께 제공하고 패키저 검증으로 확인한다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import json, hashlib, shutil, subprocess
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').exists())
OUT = ROOT / 'reports/audit_20261005'
paths = {
 'sample': OUT / 'sample/summary.json',
 'statistics': OUT / 'statistics/audit_statistics_summary.json',
 'exposure': OUT / 'exposure/audit_summary.json',
 'store_rows': OUT / 'sample/03_zero_regular_store_sales.csv',
 'availability': OUT / 'sample/05_sales_availability_by_store_size.csv',
 'support': OUT / 'statistics/identification_support.csv',
 'duration': OUT / 'exposure/duration_mix.csv',
 'service': ROOT / 'reports/07_tool_readiness_summary.json',
}
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
input_hashes = {key: sha(path) for key, path in paths.items()}
info = {key: json.loads(paths[key].read_text(encoding='utf-8')) for key in ['sample', 'statistics', 'exposure', 'service']}
for group, key in [('sample', 'source_sha256'), ('statistics', 'input_sha256')]:
    for source, digest in info[group][key].items():
        assert sha(ROOT / source) == digest, source
assert info['sample']['positive_sales_zero_regular_store_rows'] == info['sample']['positive_sales_zero_regular_store_but_positive_franchise_rows']
assert info['exposure']['monthly_overlap_audit']['missing_expected_overlap_pairs'] == 0
assert info['statistics']['direct_fe_max_coefficient_difference'] < 1e-9
print('독립 감사 입력 확인 완료. 원자료 및 기존 결과 변경 없음.')

독립 감사 입력 확인 완료. 원자료 및 기존 결과 변경 없음.


## 1. 확정 여부와 영향 범위
점포 수의 의미를 전체 점포로 설명한 것은 수정 대상이다. 표본 선택, 공연 규모, 공간 의존은 수치로 확인한 한계이지만 그로 인한 계수 편향의 방향·크기를 확정하지 않는다.

In [2]:
findings = [
 {'priority':1, 'issue':'점포 수의 정의·해석', 'status':'확정된 수정 대상', 'evidence':'원자료 919,405행에서 일반+프랜차이즈=전체; 매출양수·일반점포0인 95행은 모두 프랜차이즈 존재', 'impact':'점포당 매출·점포 수 통제 설명·0점포 이상표식', 'action':'일반·프랜차이즈·전체점포를 분리하고 매출 모집단과 맞는 분모를 확인'},
 {'priority':2, 'issue':'소규모 상권·업종의 선택적 미제공', 'status':'관측 패턴 확인', 'evidence':'전체점포 1~2개인 4,430행의 매출 제공 0%; 51개 이상은 100%', 'impact':'전체 음식점·소규모 업종으로의 일반화 제한', 'action':'자료가 제공된 상권·업종의 관련성으로 연구 범위 한정'},
 {'priority':3, 'issue':'업종별 비교 표본 차이', 'status':'비교 범위 한계 확인', 'evidence':'10종 공통표본은 16상권·1,700행으로 기본모형 7,646행의 22.23%', 'impact':'업종 특성과 지역·규모 구성 차이를 분리하기 어려움', 'action':'기본모형을 유지하고 공통표본 비교를 제한된 보조 점검으로 추가'},
 {'priority':4, 'issue':'상권 사이의 공연 공유', 'status':'노출 공유 확인; 잔차 상관 미검증', 'evidence':'500m 공연×분기당 평균 7.28개 상권에 연결; 99.30%가 여러 상권에 공유', 'impact':'상권 군집 표준오차만으로 공간 독립성을 보장할 수 없음', 'action':'잔차의 공간 상관을 확인한 뒤 공간 의존을 고려한 추론 검토'},
 {'priority':5, 'issue':'공연 ID 수와 공연 강도', 'status':'측정 한계 확인', 'evidence':'800m 내 1일 공연이 ID의 57.98%; 91일 이상 공연은 ID의 1.96%이나 달력기간의 56.16%', 'impact':'작품 개수를 실제 상연 횟수·관객 수·소비 유입으로 해석할 수 없음', 'action':'공연 ID 지표를 명확히 명명하고 기간 가중 지표를 보조 비교'},
 {'priority':6, 'issue':'적은 상권에 집중된 회귀 정보', 'status':'집중도 확인; 소표본 추론 한계', 'evidence':'중식 상위 5상권이 부분 레버리지의 53.61%; 패스트푸드 52.28%', 'impact':'명목 군집 수만으로 정밀도를 판단하기 어려움', 'action':'정보집중도·영향도와 함께 보고하고 소표본 보정을 보조 검토'},
 {'priority':7, 'issue':'역사적 공간 경계·시설 위치', 'status':'미검증', 'evidence':'현재 경계를 과거 12분기에 공통 적용; 공식 2024 공간단위 개편 공지 존재', 'impact':'같은 코드의 시점간 집계범위가 동일한지 확정 불가', 'action':'소급 재집계·경계 이력 확인; 확인 전에는 패널 단절로 단정하지 않음'},
]
display(pd.DataFrame(findings)[['priority','issue','status','action']])
pd.DataFrame(findings).to_csv(OUT / 'findings.csv', index=False, encoding='utf-8-sig')

,priority,issue,status,action
0,1,점포 수의 정의·해석,확정된 수정 대상,일반·프랜차이즈·전체점포를 분리하고 매출 모집단과 맞는 분모를 확인
1,2,소규모 상권·업종의 선택적 미제공,관측 패턴 확인,자료가 제공된 상권·업종의 관련성으로 연구 범위 한정
2,3,업종별 비교 표본 차이,비교 범위 한계 확인,기본모형을 유지하고 공통표본 비교를 제한된 보조 점검으로 추가
3,4,상권 사이의 공연 공유,노출 공유 확인; 잔차 상관 미검증,잔차의 공간 상관을 확인한 뒤 공간 의존을 고려한 추론 검토
4,5,공연 ID 수와 공연 강도,측정 한계 확인,공연 ID 지표를 명확히 명명하고 기간 가중 지표를 보조 비교
5,6,적은 상권에 집중된 회귀 정보,집중도 확인; 소표본 추론 한계,정보집중도·영향도와 함께 보고하고 소표본 보정을 보조 검토
6,7,역사적 공간 경계·시설 위치,미검증,소급 재집계·경계 이력 확인; 확인 전에는 패널 단절로 단정하지 않음


## 2. 근거를 포함한 기술 검토 보고서
공식 정의·기술 통계·추론 결과의 출처를 별도로 표시한다. 검토 보고서의 완성 상태와 기존 분석의 수정 필요 여부는 서로 다른 의미다.

In [3]:
title = '공연·외식업 분석: 추가 검토 결과'
generated = datetime.now(timezone(timedelta(hours=9))).isoformat()
sources = [{'id':key, 'label':{'sample':'자료·표본 독립 감사','statistics':'통계모형 독립 감사','exposure':'공간·공연노출 독립 감사','store_rows':'일반점포 0·매출양수 행','availability':'점포 규모별 매출 제공률','support':'업종별 회귀 정보집중도','duration':'공연 달력기간 분포','service':'분석 도구 구현 현황'}[key],
            'path':path.relative_to(ROOT).as_posix()} for key,path in paths.items()]
import sqlite3
connection = sqlite3.connect(':memory:')
for key in ['store_rows','availability','duration','support']:
    pd.read_csv(paths[key]).to_sql(key,connection,index=False,if_exists='replace')
source_queries = {
 'store_rows': 'SELECT industry, rows, franchise_positive, all_store_positive FROM store_rows',
 'availability': "SELECT *, CASE all_store_band WHEN '0' THEN 0 WHEN '1' THEN 1 WHEN '2' THEN 2 WHEN '3–5' THEN 3 WHEN '6–10' THEN 4 WHEN '11–20' THEN 5 WHEN '21–50' THEN 6 WHEN '51+' THEN 7 ELSE 8 END AS [order] FROM availability ORDER BY [order]",
 'duration': 'SELECT * FROM duration',
 'support': 'SELECT industry, area_clusters, top5_partial_leverage, concentration_effective_clusters FROM support',
}
source_queries['availability_chart'] = "SELECT * FROM (" + source_queries['availability'] + ") WHERE all_store_band NOT IN ('0','자료없음') ORDER BY [order]"
sources.append({'id':'availability_chart','label':'양의 점포 규모별 매출 제공률','path':paths['availability'].relative_to(ROOT).as_posix()})
for source in sources:
    if source['id'] in source_queries:
        table_name = 'availability' if source['id']=='availability_chart' else source['id']
        source['query'] = {'engine':'sqlite', 'language':'sql', 'sql':source_queries[source['id']], 'tables_used':[table_name], 'description':'감사 CSV를 같은 이름의 임시 SQLite 테이블에 적재한 뒤 표·차트 입력을 추출한 실제 SQL'}
sources += [{'id':'store_official','label':'서울시 점포 자료 공식 정의', 'href':'https://data.seoul.go.kr/dataList/OA-15577/S/1/datasetView.do'},
            {'id':'sales_official','label':'서울시 추정매출 자료 공식 안내', 'href':'https://data.seoul.go.kr/dataList/OA-15572/S/1/datasetView.do'}]
blocks=[]
def md(key, body, source=None):
    block={'id':key,'type':'markdown','body':body}
    if source: block['sourceId']=source
    blocks.append(block)
def table(key): blocks.append({'id':key+'_block','type':'table','tableId':key})
md('title','# '+title)
md('summary','## 검토 판단: 변수 정의를 수정하고 관련성 분석으로 제시\n\n**점포 수의 정의·해석에서 수정할 문제를 확인했다.** 일부 프랜차이즈만 있는 행을 점포가 없는데 매출이 있는 사례로 설명했고, 일반점포 수를 통제하면서 전체 점포 규모를 통제한 것처럼 읽힐 여지가 있다.\n\n기본 고정효과 모형과 보조 GEE의 독립 재계산에서는 계수·표준오차 구현 오류를 발견하지 못했다. 다만 매출 제공 표본의 선택성, 업종별 표본 차이, 공연 강도와 공간 의존 문제 때문에 결과를 전체 외식업의 인과효과나 무효과의 증명으로 제시해서는 안 된다.')
md('scope','## 검토 범위와 지표\n\n종로구·마포구의 공개 외식업 10종, 2023~2025년 12분기가 대상이다. 패널은 151상권×10업종×12분기의 18,120행이고 매출 제공은 7,661행이다. 기본 회귀는 7,646행을 사용한다. 총매출은 분기 추정매출, 야간 비중은 21~24시 매출/총매출이다. 공연 활동은 시설 점에서 현재 상권 폴리곤까지 정해진 거리 안에 있고 해당 분기와 기간이 겹치는 등록 공연 ID 수다.','sample')
md('store','## 일반점포 수를 전체점포 수로 해석한 부분은 수정해야 한다\n\n원자료 919,405행 모두에서 `유사_업종_점포_수 = 점포_수 + 프랜차이즈_점포_수`가 성립한다. 기존 `store_count`는 이 중 `점포_수`만 사용한다. 기존에 이상으로 표시한 매출양수·점포0의 95행은 모두 프랜차이즈 점포가 있었고, 전체점포까지 0인 행은 없었다.\n\n**영향 범위는 점포당 매출, 점포 수 통제 민감도 모형, 데이터 품질 설명이다.** 점포 수가 없는 기본 FE와 GEE의 수치에는 직접 영향을 주지 않는다. 매출의 모집단이 일반점포인지 전체점포인지는 추가 확인이 필요하므로 전체점포로 분모를 기계적으로 교체하는 것까지 정답으로 확정하지 않았다.','sample')
md('store_definition','서울시는 [점포 자료 안내](https://data.seoul.go.kr/dataList/OA-15577/S/1/datasetView.do)에서 전체 점포를 일반점포와 프랜차이즈의 합으로 설명한다. 일반·프랜차이즈·전체점포를 별도 변수로 명명하는 수정이 우선이다.','store_official')
table('zero_stores')
md('selection','## 작은 상권·업종은 매출 분석에서 빠지는 경향이 뚜렷하다\n\n전체점포가 1~2개인 4,430행의 매출 제공은 0%였다. 점포가 3~5개인 경우 제공률은 66.63%, 51개 이상은 100%였다. 미제공 10,459행 중 5,564행은 실제 전체점포 수가 양수였다. 단순히 미제공을 0으로 채우지 않은 것만으로 이 선택 문제가 해결되지는 않는다.\n\n아래 그림과 표는 점포 규모별 자료 관측 가능성을 보여 준다. 제공 억제나 추정 방식이 원인일 수 있지만 원인은 확정하지 않았다. 결과의 모집단을 자료가 제공된 상권·업종으로 제한해야 한다.','sample')
blocks.append({'id':'availability_chart_block','type':'chart','chartId':'availability_chart'})
table('availability')
md('comparison','## 업종 간 차이에는 서로 다른 지역 구성이 섞일 수 있다\n\n모든 10종이 같은 상권·분기에 관측되는 공통표본은 16개 상권, 170개 상권·분기, 1,700행이다. 기본모형의 22.23%만 남는다. 12분기 모두 10종이 관측되는 상권은 12개다. 현재 균형표본도 업종별로 따로 정했으므로 업종 사이의 동일 지역 비교를 보장하지 않는다.\n\n전체 표본을 공통표본으로 무조건 교체하면 대표성과 정밀도가 더 줄 수 있다. 기본 결과와 함께 작은 보조 비교를 제시하고, 업종 그 자체의 차이로 해석하는 범위를 좁히는 편이 적절하다.','sample')
md('spatial','## 같은 공연이 여러 상권에 공유되므로 공간 의존을 점검해야 한다\n\n500m 기준 고유 공연×분기 14,202조합은 평균 7.28개, 최대 22개 상권에 연결됐다. 99.30%가 2개 이상 상권에 공유됐고, 양의 공연 노출 시계열이 12분기 내내 완전히 같은 상권쌍도 21개였다. 현재 상권 군집 표준오차는 서로 다른 상권의 잔차 상관을 처리하지 않는다.\n\n**노출 공유만으로 잔차의 공간 상관이나 표준오차 편향이 입증된 것은 아니다.** 우선 잔차를 점검하고 공간 의존을 반영한 추론이 필요한지 판단해야 한다. 또 상권 면적은 약 1,869~653,127㎡로 달라 폴리곤까지 500m를 개별 음식점이나 상권 중심에서 500m로 설명하면 안 된다. 중심점이 더 올바른 거리라는 뜻도 아니다.','exposure')
md('measurement','## 공연 ID 개수는 공연 강도나 관객 수와 다르다\n\n800m 범위 공연 중 1일 공연은 ID의 57.98%이고 91일 이상 공연은 1.96%다. 그러나 시작~종료 달력기간을 합하면 후자가 56.16%를 차지한다. 지금의 지표는 이런 공연을 분기마다 각 1개 ID로 센다.\n\n아래 달력기간은 휴연을 포함할 수 있어 실제 공연일·상연 횟수로 해석할 수 없다. 공연 ID 수는 명확한 운영 지표로 유지하되 기간 가중 지표를 보조 비교하는 것이 합리적이다. 분기 자료로 관람 직전·직후의 식사·카페 이용을 직접 검증할 수 없다는 점도 명시해야 한다.','exposure')
table('duration')
md('implementation','## 회귀 계산은 재현됐지만 정보집중도는 추가로 보고해야 한다\n\n직접 더미 회귀로 재계산한 FE 계수의 최대 차이는 1.21×10⁻¹², 표준오차는 3.33×10⁻¹³이다. GEE는 저장된 상관계수 α를 고정해 독립 계산했으며 계수 차이는 5.06×10⁻¹⁴였다. α 추정 알고리즘 자체를 독립 검증한 것은 아니다. BH·Holm 보정과 10→20개 시나리오의 단위 변환도 재현됐다.\n\n반면 중식은 상위 5개 상권이 잔차화한 설명변수 정보의 53.61%, 패스트푸드는 52.28%를 차지한다. 아래 집중도 지표는 실제 자유도나 Satterthwaite 추정값이 아니다. 각 업종에서 가장 큰 부분 레버리지 상권 하나를 제외하는 동일 규칙의 진단에서는 최대 계수 변화가 원래 표준오차의 0.709배였고, 업종별 자유도 대조 후에도 BH 유의 결과는 0개였다. 큰 결론 반전의 근거는 발견하지 못했다.','statistics')
table('support')
md('method','## 검증한 모형과 해석 범위\n\n주분석은 상권×업종 및 업종×연도분기 고정효과, 상권 단위 군집 공분산을 사용한다. GEE는 Gaussian identity, within-between 분해와 상권 내 exchangeable 작업상관을 둔 보조 분석이다.\n\n업종별 기울기 20개는 BH 보정, 두 결과변수의 동일기울기 검정은 Holm 보정을 한다. 동일기울기 검정은 모든 계수가 0이라는 검정이 아니다. 기본 결과가 유의하지 않다는 사실은 관계가 없다는 증명이 아니다. 개별 95% 신뢰구간도 다중비교 동시 구간은 아니다.','statistics')
md('unverified','## 아직 확인하지 못한 점과 발견되지 않은 오류\n\n현재 경계·시설 위치의 과거 유효성과 2024년 공간단위 개편 후 자료의 소급 재집계 여부는 확인하지 못했다. [서울시 공식 안내](https://data.seoul.go.kr/dataList/OA-15572/S/1/datasetView.do)에 개편이 명시돼 있으나 이것만으로 2023/2024년 자료가 잘못 연결됐다고 단정할 수 없다.\n\n반대로 장기 공연이 시작일 검색 때문에 전부 빠졌다는 의심은 저장된 자료에서 지지되지 않았다. 예상 월별 기간 중복 36,559쌍과 실제 원문이 일치했고, 관측기간 전체를 가로지르는 장기 공연 22개가 36개월에 모두 포함됐다. 다만 수집 목록 밖 미관측 ID까지 완전하다는 증명은 아니다.')
md('service','## 서비스 구현과 통계 검증은 구분해야 한다\n\n현재 07번은 저장된 결과와 지표를 조회하는 오프라인 도구다. LLM·자연어 계획·웹 서비스는 연결되지 않았고, RDF도 총매출과 공연 관계 일부를 담는다. 따라서 현 단계에서 Agentic AI 서비스의 효과나 온톨로지의 오류 감소를 검증했다고 쓰면 안 된다. 기존 구현 현황 표시는 이 범위를 명시하고 있다.','service')
md('next','## 우선순위\n\n1. **점포 정의부터 수정한다.** 일반·프랜차이즈·전체 점포를 구분하고 95행의 설명을 바로잡는다. 매출 모집단을 확인한 뒤 점포당 매출과 점포 통제 민감도 모형을 재검토한다.\n2. **논문의 해석 범위를 좁힌다.** 관측된 상권·업종의 분기 관련성으로 기술하고, 인과효과·전체 외식업·실제 관객 유입으로 확대하지 않는다.\n3. **필요한 보조 검증만 추가한다.** 공통표본, 공간 잔차·소표본 추론, 기간 가중 노출을 사전에 정한 제한된 비교로 점검한다. 유의한 결과가 나오는 설정을 고르지 않는다.\n4. **남은 확인 질문을 분리한다.** 매출이 포괄하는 점포 모집단, 소규모 행 미제공 기준, 과거 경계·시설 위치 이력이 핵심이다. 이번 검토는 본 분석을 덮어쓰지 않고 별도 감사 자료로 남겼다.')
frames = {key:pd.read_sql_query(source_queries[source],connection) for key,source in {'zero_stores':'store_rows','availability':'availability','duration':'duration','support':'support'}.items()}
labels={
 'zero_stores': [('industry','업종'),('rows','매출양수·일반점포0 행'),('franchise_positive','프랜차이즈 존재 행'),('all_store_positive','전체점포 양수 행')],
 'availability':[('order','순서'),('all_store_band','전체점포 수'),('expected_rows','행 수'),('sales_rows','매출 제공 행'),('availability_pct','제공률(%)')],
 'duration':[('duration_group','달력기간'),('performance_ids','공연 ID 수'),('id_share_pct','ID 비중(%)'),('envelope_share_pct','달력기간 비중(%)')],
 'support':[('industry','업종'),('area_clusters','명목 상권 수'),('top5_partial_leverage','상위5개 부분 레버리지'),('concentration_effective_clusters','집중도 기반 유효 군집 수')],
}
source_map={'zero_stores':'store_rows','availability':'availability','duration':'duration','support':'support'}
table_titles={'zero_stores':'프랜차이즈만 존재하는 양매출 관측','availability':'점포 규모별 매출 제공','duration':'공연 기간과 ID 비중','support':'업종별 회귀 정보집중도'}
tables=[]
for key, columns in labels.items():
    tables.append({'id':key,'title':table_titles[key],'dataset':key,'sourceId':source_map[key],
        'defaultSort':{'field': 'order' if key=='availability' else ('concentration_effective_clusters' if key=='support' else ('rows' if key=='zero_stores' else 'performance_ids')),
                       'direction':'asc' if key in ['availability','support'] else 'desc'},
        'columns':[{'field':col,'label':label,**({'type':'text'} if col in ['industry','all_store_band','duration_group'] else {'format':'percent'} if col=='top5_partial_leverage' else {'format':'number'})} for col,label in columns]})
frames['availability_chart'] = pd.read_sql_query(source_queries['availability_chart'],connection)
connection.close()
charts=[{'id':'availability_chart','title':'점포 규모별 매출 제공률','subtitle':'2023~2025년 · 상권×업종×분기 · 전체점포 수 확인 행','type':'bar','dataset':'availability_chart','sourceId':'availability_chart','valueFormat':'number','encodings':{'x':{'field':'all_store_band','type':'nominal','label':'전체점포 수'},'y':{'field':'availability_pct','type':'quantitative','label':'매출 제공률(%)'},'tooltip':[{'field':'expected_rows','type':'quantitative','label':'행 수'},{'field':'sales_rows','type':'quantitative','label':'매출 제공 행'}]}}]
artifact={'surface':'report', 'manifest':{'version':1,'surface':'report','title':title,'generatedAt':generated,
    'description':'확정된 수정 대상과 설계 한계를 구분한 독립 검토', 'blocks':blocks,'tables':tables,'charts':charts,'sources':sources},
    'snapshot':{'version':1,'generatedAt':generated,'status':'ready','datasets':{key:json.loads(frame.to_json(orient='records',force_ascii=False)) for key,frame in frames.items()},'accessIssues':[]},
    'sources':sources}
(OUT/'artifact.json').write_text(json.dumps(artifact,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
(OUT/'report_inputs.json').write_text(json.dumps({'input_sha256':input_hashes,'delivery':'portable_html','audience':'technical','table_rationale':'정확한 감사 수치와 분모 비교','chart_contract':'7개 전체점포 규모별 매출 제공률; 단일계열 막대; 0~100%; 공개 상권분기 행 기준; 미관측 점포 규모 별도 표','analysis_assessment':'수정 필요; 기본 회귀 계산 오류 미발견'},ensure_ascii=False,indent=2),encoding='utf-8')
assert all(sha(paths[key])==digest for key,digest in input_hashes.items())
print('검토 보고서 입력 저장:',(OUT/'artifact.json').relative_to(ROOT))

검토 보고서 입력 저장: reports\audit_20261005\artifact.json


In [4]:
builder = Path(r'C:/Users/mamap/.codex/plugins/cache/openai-curated-remote/data-analytics/0.2.10-13ceeea1f599/skills/build-report/scripts/deliver_portable_artifact.mjs')
result = subprocess.run([shutil.which('node'), str(builder), '--input', str(OUT/'artifact.json'), '--output', str(OUT/'report.html')], capture_output=True, text=True, encoding='utf-8', timeout=120)
print(result.stdout[-6000:])
if result.returncode: print(result.stderr[-6000:])
assert result.returncode == 0, '보고서 패키징 검증 실패'
(OUT/'report_delivery_receipt.json').write_text(result.stdout, encoding='utf-8')

{"ok":true,"html":"C:\\Coding\\졸업프로젝트\\reports\\audit_20261005\\report.html","stages":{"validation":"passed","package":"passed","verification":"structural_only"},"browserWarning":{"code":"browser_unavailable","message":"No installed Chromium headless-shell executable was found. Set CHROMIUM_EXECUTABLE_PATH (or the legacy PLAYWRIGHT_EXECUTABLE_PATH) or preinstall Chromium headless-shell under PLAYWRIGHT_BROWSERS_PATH; portable tooling never downloads browsers."},"counts":{"blocks":19,"charts":1,"html":0,"metrics":0,"tables":4},"sourceDialog":"not_verified","sourceInteraction":"not_verified","viewports":[],"timings":{"validateAndPackageMs":122.9,"staticChartExtractionMs":48.6,"structuralVerificationMs":3.9,"totalMs":136.4}}



732

**해석:** 새 감사 자료는 문제의 유무와 영향 범위를 보여 준다. 기존 분석을 새로운 방법으로 교체하거나 연구 가설을 사후 선택한 결과가 아니다. 표본·변수 정의와 남은 질문을 검토한 뒤 필요한 부분만 수정한다.